In [5]:
import pandas as pd
df = pd.read_csv('modified_digital_media_with_business_segment.csv')
common_drop = ['campaign_id', 'date', 'performance_segment']

# Regression
X_reg = df.drop(columns=['revenue_usd', 'performance_tier_index'] + common_drop, errors='ignore')
y_reg = df['revenue_usd']

# Classification
X_clf = df.drop(columns=['performance_tier_index', 'revenue_usd'] + common_drop, errors='ignore')
y_clf = df['performance_tier_index']

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

cat_cols = X_reg.select_dtypes(include='object').columns.tolist()
num_cols = X_reg.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ("num", "passthrough", num_cols)
])

In [8]:
from xgboost import XGBRegressor

reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        subsample=0.6,
        reg_lambda=2,
        reg_alpha=1,
        n_estimators=1500,
        min_child_weight=3,
        max_depth=2,
        learning_rate=0.01,
        gamma=0.5,
        random_state=42,
        objective='reg:squarederror'
    ))
])

reg_pipeline.fit(X_reg, y_reg)

import joblib
joblib.dump(reg_pipeline, "models/reg_pipeline.pkl")

['models/reg_pipeline.pkl']

In [9]:
from xgboost import XGBClassifier

clf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        subsample=1.0,
        reg_lambda=2,
        reg_alpha=1,
        n_estimators=1000,
        min_child_weight=3,
        max_depth=2,
        learning_rate=0.02,
        gamma=0.5,
        random_state=42,
        eval_metric='mlogloss'
    ))
])

clf_pipeline.fit(X_clf, y_clf)

joblib.dump(clf_pipeline, "models/clf_pipeline.pkl")

['models/clf_pipeline.pkl']